# 02 Paradox Detection

Author: Endri Sojeva

This notebook compares causal cycles and consistency violations across four reproducible scenarios.

## Setup

Use the project's Python environment. In Kaggle, the notebook loads the package and the same scenario files from a source snapshot tied to a GitHub commit.

In [ ]:
import platform
import subprocess
from importlib.metadata import version
from io import BytesIO
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display
from timetravel.algorithms.dfs import find_cycle
from timetravel.algorithms.tarjan import strongly_connected_components
from timetravel.detection import detect_paradoxes
from timetravel.scenarios import load_scenario
from timetravel.visualization import draw_timeline

source_commit = globals().get("SOURCE_COMMIT")
if source_commit is None:
    try:
        source_commit = subprocess.check_output(
            ["git", "rev-parse", "HEAD"], text=True, stderr=subprocess.DEVNULL
        ).strip()
    except (OSError, subprocess.CalledProcessError):
        source_commit = "local checkout (commit unavailable)"
print({"source_commit": source_commit, "python": platform.python_version(),
       **{name: version(name) for name in ("networkx", "matplotlib", "pandas")}})

examples = globals().get("SCENARIO_DIR")
if examples is None:
    examples = next(directory / "data/examples" for directory in (Path.cwd(), Path.cwd().parent)
                    if (directory / "data/examples").is_dir())

## Structural evidence: DFS and Tarjan SCC

DFS explores the outgoing edges of each vertex. An edge to a vertex on the active path closes a cycle, including a self-loop. Returning the closed path gives a verifiable witness without enumerating all cycles.

Tarjan uses discovery indices, low-link values and a stack of unassigned vertices. When a vertex's low-link equals its index, its SCC can be popped. Each multi-vertex SCC is cyclic, as is a singleton with a self-loop. The report finds one witness in each cyclic SCC.

In [ ]:
sample = load_scenario(examples / "consistent_loop.json")
print("DFS example:", find_cycle(sample.graph))
print("Tarjan SCCs:", strongly_connected_components(sample.graph))

## Four consistency outcomes

A timeline is consistent if the selected rules find no violations. The default rules enforce compatible state assertions at one time, occurred sources for occurred effects, and occurred endpoints for active named trips. Scenarios can add explicit event prerequisites.

In the grandfather case, a traveler prevents the ancestor's survival. The recorded birth and departure still rely on that survival. The failure is in those dependencies. The structural cycle supplies separate evidence.

In [ ]:
cases = {
    "valid_timeline": (True, 0, 0, 0),
    "consistent_loop": (True, 1, 0, 0),
    "contradictory_timeline": (False, 0, 1, 0),
    "grandfather_paradox": (False, 1, 1, 1),
}
rows, reports, scenarios = [], {}, {}
for name, expected in cases.items():
    scenario = scenarios[name] = load_scenario(examples / f"{name}.json")
    report = reports[name] = detect_paradoxes(scenario.graph, scenario.constraints)
    result = (report.consistent, len(report.cycles), len(report.logical_violations),
              len(report.causal_violations))
    assert result == expected, (name, result)
    rows.append((name, *result))

summary = pd.DataFrame(rows, columns=["scenario", "consistent", "cycle witnesses", "logical", "causal"])
display(summary)
print(summary.to_string(index=False))
for issue in reports["grandfather_paradox"].violations:
    print(issue.rule, issue.event_ids, issue.relations, issue.explanation)

## Compare the graphs

Purple edges show cycle witnesses. Red nodes appear in rule violations. The acyclic beacon example has red nodes and no purple edges.

In [ ]:
for name in ("consistent_loop", "contradictory_timeline", "grandfather_paradox"):
    report = reports[name]
    figure = draw_timeline(
        scenarios[name].graph,
        problem_nodes={event for violation in report.violations for event in violation.event_ids},
        cycle_edges={edge for witness in report.cycles for edge in witness.relations},
    )
    figure.axes[0].set_title(name.replace("_", " ").title())
    with BytesIO() as buffer:
        figure.savefig(buffer, format="png", bbox_inches="tight")
        display(Image(data=buffer.getvalue()))
    plt.close(figure)

## Complexity and interpretation

DFS and Tarjan each run in O(V + E) time and O(V) auxiliary space. Witnesses from disjoint cyclic SCCs retain the same aggregate bound. Rule evaluation additionally reads asserted state, recorded trips and declared prerequisites. The number of simple cycles can be exponential in the graph size.

The four assertions check correctness for the selected fixtures. They show why cyclicity and consistency must be reported separately. The grandfather scenario shows one logical and one causal violation for distinct declared dependencies. These are functional demonstrations. Large-scale benchmark measurements are reserved for a later notebook.